### RAG PIPELINE

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

CHROMA_PATH     = "../data/processed/chroma_db"
COLLECTION_NAME = "elte_ik"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
TOP_K           = 3

In [2]:
client     = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection(name=COLLECTION_NAME)  
model      = SentenceTransformer(EMBEDDING_MODEL)         
print(f"Collection has {collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection has 6103 documents


In [3]:
def retrieve(query: str, n_results: int = TOP_K) -> list[dict]:
    query_emb = model.encode([query]).tolist()
    results   = collection.query(query_embeddings=query_emb, n_results=n_results)
    return [
        {"content": doc, "metadata": meta}
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]

In [5]:
hits = retrieve("show me the bsc curriculum?")
for i, h in enumerate(hits):
    print(f"\n[{i+1}] chunk {h['metadata']['chunk_id']} — {h['metadata']['file_name']}")
    print(h['content'][:300])


[1] chunk 4162 — The curriculum.pdf
Understanding the curriculum
For BSc students
It is vital for you to understand your curriculum. Here is what you need to know:
Key terms
▪
Compulsory subject: They can be lectures or group work (seminars or practices) and you must complete all of them in order
to graduate.
▪
Compulsory Elective sub

[2] chunk 4653 — curriculum.html
For Students
Official curriculum files
Equivalencies
♦ Essential guide - Curriculum ♦
It is vital for you to understand your curriculum. Here is what you need to know:
Key terms
Compulsory subject: They can be lectures or group work (seminars or practices) and you must complete all them in order to 

[3] chunk 1072 — Conditions for obtaining Diploma.pdf
Understanding the curriculum:
MSc in Geoinformatics (120 credits)
Compulsory subjects 70 cr.
Compulsory elective subjects
(A modul/ B modul) 24 cr.
Elective subjects 6 cr.
Thesis 20 cr.


In [ ]:
def build_prompt(query: str, chunks: list[dict]) -> str:
    context = "\n\n".join(
        f"[Source: {c['metadata']['file_name']}]\n{c['content']}"
        for c in chunks
    )
    return f"""You are a helpful assistant for ELTE Faculty of Informatics students.
Answer the question using only the context below. If the answer is not in the context, say so.

Context:
{context}

Question: {query}
Answer:"""


In [6]:
import requests

OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2:3b"
TEMPERATURE  = 0.1    # low = factual, less hallucination; high = creative
TIMEOUT_S    = 120    # Ollama slow on first model load

In [ ]:
def check_ollama() -> bool:
    try:
        r = requests.get("http://localhost:11434", timeout=5)
        return r.status_code == 200
    except requests.exceptions.ConnectionError:
        return False

def call_ollama(prompt: str, model: str = OLLAMA_MODEL) -> str:
    if not check_ollama():
        raise RuntimeError("Ollama is not running. Start it with: ollama serve")

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "num_ctx": 2048,  
        }
    }

    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT_S)
        r.raise_for_status()
        return r.json()["response"].strip() 
    except requests.exceptions.Timeout:
        raise RuntimeError(f"Ollama timed out after {TIMEOUT_S}s — model may still be loading")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"Ollama HTTP {e.response.status_code} — is model '{model}' pulled?")
    except KeyError:
        raise RuntimeError(f"Unexpected Ollama response: {r.text[:200]}")

In [8]:
def rag_query(query: str) -> dict:
    chunks = retrieve(query)
    prompt = build_prompt(query, chunks)
    answer = call_ollama(prompt)
    return {
        "answer": answer,
        "sources": [
            {"chunk_id": c["metadata"]["chunk_id"], "file": c["metadata"]["file_name"]}
            for c in chunks
        ]
    }

# End-to-end test
result = rag_query("What happens if I don't complete a weak prerequisite?")
print("Answer:", result["answer"])
print("\nSources:")
for s in result["sources"]:
    print(f"  chunk {s['chunk_id']} — {s['file']}")

Answer: If you don't complete a weak prerequisite, your grade in the follow-up subject will automatically be failed/unfulfilled, even if you pass its exam.

Sources:
  chunk 8 — The prerequisites.pdf
  chunk 6 — The prerequisites.pdf
  chunk 10 — The prerequisites.pdf
